# Rank + Tuned Ensemble + Multi-scale Residual Lens + Dependence/Early-onset Lens (E62 = `b_no_accf`)

Extends the shipped **trail120** champion (LB 61.58%) with a 10-feature dependence / early-onset residual bank (fresh-confirmed **+0.0066, 4/4 seeds**): AR-studentized residual + early-onset CUSUM + ARCH contrast + sup-F scan. The ACF-lag contrasts were noise and are dropped. Everything else (2-member fixed-CDF blend, `w=0.6`, tuned HGB + `rank_xendcg`) is unchanged.


Same E36 2-member ensemble (tuned HGB + `rank_xendcg`, fixed-CDF blend `w=0.6`) as the LB 61.32%
champion, with the residual lens extended by a **second reference span**.

## Residual lens (E54: two reference spans)

A structural break shows up first as the **reference model's residuals blowing up**. Fit `AR8` on the
historical segment, apply it forward onto trailing sub-windows (60/130/250), and add
`log( var(sub-window residuals) / var(reference-fit residuals) )`.

- **E53 (LB 61.32%, shipped champion):** one reference — AR8 fit on the **full** history (+3 feats).
- **E54 (this notebook):** add a **second reference — AR8 fit on only the last 120 historical points**
  (`REF_TRAILS = (None, 120)`, +6 feats total). The recent-history dynamics are closer to the break
  region and carry *orthogonal* signal. Fresh-confirmed **+0.0303** (min_t 2.89, 4/4) vs the champion's
  +0.0232 — a ~+0.007 marginal gain over E53.

Both champion members consume the block (via the `compute_features_ms` wrap). `STEPS_PER_SERIES = 30`.
Config knobs: `AR_ORDERS`, `RESIDUAL_SCALES`, `REF_TRAILS`, `RESIDUAL_MODE`.


## Getting set up

Install the Crunch CLI and initialize the local environment (downloads the data and wires up the
tooling). Get your own token from the competition page and paste it below.

>> https://hub.crunchdao.com/competitions/structural-break-real-time/submit/notebook

In [ ]:
%pip install crunch-cli lightgbm --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time W0gwRvfw4tC1dKElMGK9SzjR

In [ ]:
import os

# Keep BLAS / OpenMP single-threaded. infer() runs in several parallel
# processes, and single-row model predictions are dramatically faster without
# per-call thread-pool spin-up. Env vars only work if set before numpy loads
# (not guaranteed on the cloud), so infer() ALSO caps threads at call time via
# threadpool_limits below.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import math
from collections import deque
from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from threadpoolctl import threadpool_limits
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score

In [ ]:
import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()

In [ ]:
# Load the data (only needed for local exploration / testing).
train_data, test_data = crunch_tools.load_data()
print(f"train series: {len(train_data)}  |  test series: {len(test_data)}")

## Feature extractor

Shared by `train()` and `infer()` so the training features and the streaming features are computed
identically. `hist_context()` summarizes the historical segment once per series; `compute_features()`
turns a trailing online window into the 15-dim feature vector.

In [ ]:
FEATURE_NAMES = [
    "mean_z", "log_var_ratio", "ks", "skew_diff", "kurt_diff",
    "ac1_diff", "ac2_diff", "iqr_ratio", "tail2", "tail3",
    "zero_cross", "slope_norm", "spec_centroid_diff", "rng_ratio",
    "spec_entropy",
]


def _autocorr(a, lag):
    if len(a) <= lag:
        return 0.0
    x = a[:-lag]
    y = a[lag:]
    xm = x.mean()
    ym = y.mean()
    denom = math.sqrt(((x - xm) ** 2).sum() * ((y - ym) ** 2).sum())
    if denom < 1e-12:
        return 0.0
    return float(((x - xm) * (y - ym)).sum() / denom)


def _spectral_centroid(a):
    a = a - a.mean()
    sp = np.abs(np.fft.rfft(a))
    freqs = np.fft.rfftfreq(len(a))
    s = sp.sum()
    if s < 1e-12:
        return 0.0
    return float((freqs * sp).sum() / s)


def _skew_kurt(w, mean_w, pop_var):
    """Population skew and excess kurtosis from central moments (pure numpy)."""
    if pop_var < 1e-12:
        return 0.0, 0.0
    d = w - mean_w
    m3 = float(np.mean(d ** 3))
    m4 = float(np.mean(d ** 4))
    sk = m3 / (pop_var ** 1.5)
    ku = m4 / (pop_var ** 2) - 3.0
    return sk, ku


def _fast_ks(w_sorted, h_sorted):
    """Two-sample KS statistic from pre-sorted arrays (numpy searchsorted)."""
    h = len(h_sorted)
    n = len(w_sorted)
    allv = np.concatenate([h_sorted, w_sorted])
    cdf_h = np.searchsorted(h_sorted, allv, side="right") / h
    cdf_w = np.searchsorted(w_sorted, allv, side="right") / n
    return float(np.max(np.abs(cdf_h - cdf_w)))


def hist_context(x_h, hist_cap):
    """Precompute historical-segment statistics + a sorted sample for the KS test."""
    n = len(x_h)
    mu = float(x_h.mean()) if n else 0.0
    var = float(x_h.var(ddof=1)) if n > 1 else 1.0
    var = max(var, 1e-12)
    sd = math.sqrt(var)
    pop_var = float(x_h.var()) if n else 1.0
    sk, ku = _skew_kurt(x_h, mu, pop_var) if n > 2 else (0.0, 0.0)
    ac1 = _autocorr(x_h, 1)
    ac2 = _autocorr(x_h, 2)
    q25, q75 = np.quantile(x_h, [0.25, 0.75]) if n else (0.0, 0.0)
    iqr = float(q75 - q25)
    iqr = iqr if iqr > 1e-12 else 1e-12
    rng = float(x_h.max() - x_h.min()) if n else 1.0
    rng = rng if rng > 1e-12 else 1e-12
    sc = _spectral_centroid(x_h) if n > 4 else 0.0
    if n > hist_cap:
        stride = n // hist_cap
        sample = x_h[::stride][:hist_cap]
    else:
        sample = x_h
    sample_sorted = np.sort(np.asarray(sample, dtype=np.float64))
    return {
        "mu": mu, "sd": sd, "var": var, "skew": sk, "kurt": ku,
        "ac1": ac1, "ac2": ac2, "iqr": iqr, "rng": rng, "sc": sc,
        "sample_sorted": sample_sorted,
    }


def compute_features(w, ctx):
    """w: 1-D np.array trailing window. ctx: hist_context dict. -> feature list (len 15).

    Pure-numpy (no scipy.stats) so it is fast enough for per-step streaming inference.
    """
    n = len(w)
    mean_w = float(w.mean())
    var_w = max(float(w.var()), 1e-12)
    se = ctx["sd"] / math.sqrt(n)
    mean_z = (mean_w - ctx["mu"]) / max(se, 1e-8)
    log_var_ratio = math.log(var_w / ctx["var"])
    w_sorted = np.sort(w)
    ks = _fast_ks(w_sorted, ctx["sample_sorted"])
    sk, ku = _skew_kurt(w, mean_w, var_w)
    skew_diff = sk - ctx["skew"]
    kurt_diff = ku - ctx["kurt"]
    ac1_diff = _autocorr(w, 1) - ctx["ac1"]
    ac2_diff = _autocorr(w, 2) - ctx["ac2"]
    q25 = w_sorted[int(0.25 * (n - 1))]
    q75 = w_sorted[int(0.75 * (n - 1))]
    iqr_w = max(float(q75 - q25), 1e-12)
    iqr_ratio = iqr_w / ctx["iqr"]
    dev = np.abs(w - ctx["mu"]) / ctx["sd"]
    tail2 = float((dev > 2.0).mean())
    tail3 = float((dev > 3.0).mean())
    centered = w - ctx["mu"]
    zero_cross = float((np.diff(np.sign(centered)) != 0).mean()) if n > 1 else 0.0
    i = np.arange(n, dtype=np.float64)
    sx = i.sum()
    sxx = (i * i).sum()
    sy = w.sum()
    sxy = (i * w).sum()
    denom = n * sxx - sx * sx
    slope = (n * sxy - sx * sy) / denom if abs(denom) > 1e-12 else 0.0
    slope_norm = slope * n / ctx["sd"]
    spec_centroid_diff = _spectral_centroid(w) - ctx["sc"]
    rng_ratio = float(w_sorted[-1] - w_sorted[0]) / ctx["rng"]
    # spectral entropy of the window (E13: robust +0.0050 TS-AUC @ WINDOW=130, t=+4.67)
    ps = np.abs(np.fft.rfft(w - mean_w)) ** 2
    ps_sum = ps.sum()
    if ps_sum < 1e-12 or len(ps) < 2:
        spec_entropy = 0.0
    else:
        p = ps / ps_sum
        p = p[p > 0]
        spec_entropy = float(-(p * np.log(p)).sum()) / math.log(len(ps))
    return [
        mean_z, log_var_ratio, ks, skew_diff, kurt_diff,
        ac1_diff, ac2_diff, iqr_ratio, tail2, tail3,
        zero_cross, slope_norm, spec_centroid_diff, rng_ratio,
        spec_entropy,
    ]


def compute_features_ms(w, ctx):
    """Multi-scale wrapper (E17 winner ms_60_130_250, 45 feats, 10-fold t=+3.06).

    Compute the base15 feature set on nested trailing sub-windows w[-s:] for each
    scale s in MS_SCALES and concatenate. Short scales are responsive; long scales
    are slow -> the model sees several views of the same series-vs-history contrast.
    Guarded so a scale larger than the current window just reuses the full window.
    """
    n = len(w)
    out = []
    for s in MS_SCALES:
        ws = w if s >= n else w[-s:]
        out.extend(compute_features(ws, ctx))
    return out


## Configuration

`WINDOW`, `MIN_N`, `HIST_CAP` shape the features; `PREDICT_EVERY` throttles inference cost;
`STEPS_PER_SERIES` / `SEED` control training-set sampling.

In [ ]:
# @crunch/keep:on
WINDOW           = 250   # buffer = largest MS scale (E17 multi-scale)
MS_SCALES        = (60, 130, 250)  # E17 champion ms_60_130_250 (45 feats); shared by both models
MIN_N            = 15    # warm-up: emit 0.0 until this many online points seen
HIST_CAP         = 200   # max historical points sampled for the KS test
PREDICT_EVERY    = 1     # recompute features + prediction every N steps (1 = no staleness)
STEPS_PER_SERIES = 10    # sampled online steps per training series for the HGB (pointwise)
DENSE_K          = 60    # sampled online steps per series for the ranker (dense trajectory)
MAX_TRAIN_SERIES = 10**9 # consume ALL series the platform provides (no cap)
MAX_ITER         = 400   # E32 tuned HGB winner
SEED             = 42
RANK_SEED        = 42    # ranker model seed
BLEND_W          = 0.6   # score = w*pct_hgb + (1-w)*pct_rank; confirmed w=0.6 (fresh seeds, t=+3.07)

# LGBM listwise ranker (E33/E36): rank_xendcg, query group = online step t
LGB_PARAMS = dict(
    objective="rank_xendcg", n_estimators=300, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, reg_lambda=1.0, subsample=1.0, colsample_bytree=0.7,
    random_state=RANK_SEED, verbose=-1,
)

In [ ]:
# @crunch/keep:on
# =====================================================================================
# Residual-variance-ratio feature block  (E53 multi-scale + E54 trailing-history reference)
# -------------------------------------------------------------------------------------
# A structural break shows up first as the REFERENCE model's residuals blowing up. Fit AR(p) on the
# historical segment, apply forward onto trailing SUB-windows of the online buffer, and use
# log( var(sub-window residuals) / var(reference-fit residuals) ) at each scale.
# E54: fit the reference on TWO history spans -- the FULL history AND only the LAST 120 points. The
# recent-history reference (dynamics closer to the break region) adds ORTHOGONAL signal on top of the
# full-history one. Fresh-confirmed +0.0303 (min_t 2.89, 4/4) vs +0.0232 for the full-history-only block.
#
#   feats = |REF_TRAILS| * |AR_ORDERS| * |RESIDUAL_SCALES|   (min mode: var-ratio only)
# =====================================================================================
RESIDUAL_MODE    = "min"
AR_ORDERS        = (8,)                # the AR8 residual is the sharpest single reference
RESIDUAL_SCALES  = (60, 130, 250)      # multi-scale residual (mirrors the champion MS_SCALES)
REF_TRAILS       = (None, 120)         # None = full history; 120 = last 120 historical points (E54)
STEPS_PER_SERIES = 30                  # secondary confirmed lever; set back to 10 to disable


def _fit_ar(x, p):
    """AR(p) least-squares fit on an ordered series -> (coef[p+1], residual variance)."""
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < p + 8:
        return None
    cols = [x[p - lag:n - lag] for lag in range(1, p + 1)]
    Xd = np.column_stack(cols + [np.ones(n - p)])
    coef, *_ = np.linalg.lstsq(Xd, x[p:], rcond=None)
    return coef, float(np.var(x[p:] - Xd @ coef)) + 1e-12


def _ar_resid(w, coef, p):
    n = len(w)
    if n < p + 3:
        return None
    cols = [w[p - lag:n - lag] for lag in range(1, p + 1)]
    return w[p:] - np.column_stack(cols + [np.ones(n - p)]) @ coef


def _acf(x, k):
    if len(x) <= k:
        return 0.0
    x0 = x - x.mean()
    d = float(np.dot(x0, x0)) + 1e-12
    return float(np.dot(x0[k:], x0[:-k])) / d


def _fit_ar_ref(x_h):
    """AR fits per (trail-span, order). trail=None uses the full history."""
    x = np.asarray(x_h, dtype=np.float64)
    ref = {}
    for trail in REF_TRAILS:
        xr = x if trail is None else x[-trail:]
        ref[trail] = {p: _fit_ar(xr, p) for p in AR_ORDERS}
    return ref


def _resid_stats(r, refvar):
    rv = float(np.var(r)) + 1e-12
    varratio = math.log(rv / refvar)
    if RESIDUAL_MODE == "min":
        return [varratio]
    sig = math.sqrt(refvar)
    n = len(r)
    q = 0.0
    for k in (1, 2, 3):
        if n - k > 0:
            ak = _acf(r, k)
            q += ak * ak / (n - k)
    cs = np.cumsum(r) / (sig * math.sqrt(n))
    return [varratio, float(np.mean(r)) / math.sqrt(rv), _acf(r, 1),
            n * (n + 2) * q, float(np.max(np.abs(cs))), float(np.mean(np.abs(r) > 3.0 * sig))]


def _residual_feats(w, ar_ref):
    w = np.asarray(w, dtype=np.float64)
    per = 1 if RESIDUAL_MODE == "min" else 6
    out = []
    for trail in REF_TRAILS:
        for p in AR_ORDERS:
            m = ar_ref[trail].get(p)
            for s in RESIDUAL_SCALES:
                ws = w if s >= len(w) else w[-s:]
                r = _ar_resid(ws, m[0], p) if m is not None else None
                out += ([0.0] * per) if (r is None or len(r) < 5) else _resid_stats(r, m[1])
    return out


# Wrap the champion's context + feature builders so the residual block rides along everywhere.
# train() and infer() call hist_context / compute_features_ms BY NAME, so no other change is needed.
_base_hist_context = hist_context
_base_compute_features_ms = compute_features_ms


def hist_context(x_h, hist_cap):
    ctx = _base_hist_context(x_h, hist_cap)
    ctx["ar_ref"] = _fit_ar_ref(x_h)
    return ctx


def compute_features_ms(w, ctx):
    return _base_compute_features_ms(w, ctx) + _residual_feats(w, ctx["ar_ref"])


_n_extra = len(REF_TRAILS) * len(AR_ORDERS) * len(RESIDUAL_SCALES) * (1 if RESIDUAL_MODE == "min" else 6)
print(f"residual block: mode={RESIDUAL_MODE}, AR{AR_ORDERS} x trails{REF_TRAILS} x scales{RESIDUAL_SCALES} "
      f"-> +{_n_extra} feats; STEPS_PER_SERIES={STEPS_PER_SERIES}")


## Dependence / early-onset residual block (E62 = `b_no_accf`)

Fresh-confirmed **+0.0066 (4/4 seeds)** on top of the trail120 champion — the best additive
lead since trail120. Targets the two axes the EDA flagged as the residual lens's blind spots:

- **Latency** on pure dependence-structure breaks (`ma` / `ar_coef`, detected ~120–190 pts late)
  → an **early-onset residual CUSUM** (accumulate residual surprise from the first post-break point).
- **Specificity leak** (strong-AR quiet series false-positive) → the **AR-studentized residual**
  (z-score the window log-var-ratio against the series' *own* sliding-window historical null).

Plus an ARCH-lite squared-residual contrast and a sup-F / Chow max-split scan. The ACF-lag
contrasts from the full 14-feature bank were pure noise (`b_accf`) and are **dropped** here → the
10-feature `b_no_accf` set. Rides along everywhere via the same `hist_context` / `compute_features_ms`
wrapping the residual block uses, so `train()` / `infer()` need no change.

In [ ]:
# @crunch/keep:on
# =============================================================================
# Dependence / early-onset residual block  (E62: EDA levers 1 + 2 -> b_no_accf)
# -----------------------------------------------------------------------------
# +10 causal features on top of the 51-col trail120 champion (fresh-confirmed +0.0066, 4/4):
#   [0] logvr        log AR8-residual variance ratio (window vs full-history reference)
#   [1] z_student    (logvr - hist_mean) / hist_sd  -- AR-studentized (specificity fix, lever 2)
#   [2..5] cusum_*   early-onset residual CUSUM  max / end / argmax-pos / drift  (lever 1)
#   [6] arch_contrast  lag-1 autocorr of squared residuals, window vs history (ARCH/BDS proxy)
#   [7..9] supf_*    sup-F / Chow max-split scan inside the window  t / |log var-ratio| / pos
# The 4 ACF-lag contrasts (dacf1/2/5/10) from the full 14-feat bank were noise -> DROPPED.
# =============================================================================
_HIST_NULL_WINS = 24          # historical windows used to estimate the per-series logVR null
_DEP_EPS = 1e-12
_DEP_KEEP = [0, 1, 2, 3, 4, 5, 10, 11, 12, 13]   # drop dacf1/2/5/10 (indices 6..9)


def _dep_autocorr(a, lag):
    if len(a) <= lag:
        return 0.0
    x = a[:-lag]
    y = a[lag:]
    xm = x.mean()
    ym = y.mean()
    denom = math.sqrt(((x - xm) ** 2).sum() * ((y - ym) ** 2).sum())
    if denom < 1e-12:
        return 0.0
    return float(((x - xm) * (y - ym)).sum() / denom)


def _dep_fit_ar(x, p):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < p + 8:
        return None
    cols = [x[p - lag:n - lag] for lag in range(1, p + 1)]
    Xd = np.column_stack(cols + [np.ones(n - p)])
    coef, *_ = np.linalg.lstsq(Xd, x[p:], rcond=None)
    return coef, float(np.var(x[p:] - Xd @ coef)) + 1e-12


def _dep_ar_resid(w, coef, p):
    n = len(w)
    if n < p + 3:
        return None
    cols = [w[p - lag:n - lag] for lag in range(1, p + 1)]
    return w[p:] - np.column_stack(cols + [np.ones(n - p)]) @ coef


def _dep_fit_refs(x_h):
    x = np.asarray(x_h, dtype=np.float64)
    m8 = _dep_fit_ar(x, 8)
    coef, refvar = (m8[0], m8[1]) if m8 is not None else (None, 1.0)
    logvr = []
    arch_vals = []
    if coef is not None and len(x) >= 300:
        win = 250
        starts = np.linspace(0, len(x) - win, _HIST_NULL_WINS).astype(int)
        for s in starts:
            seg = x[s:s + win]
            r = _dep_ar_resid(seg, coef, 8)
            if r is None or len(r) < 20:
                continue
            logvr.append(math.log((float(np.var(r)) + _DEP_EPS) / refvar))
            arch_vals.append(_dep_autocorr(r * r, 1))
    null = (float(np.mean(logvr)), float(np.std(logvr) + _DEP_EPS)) if len(logvr) >= 3 else (0.0, 1.0)
    arch_hist = float(np.mean(arch_vals)) if arch_vals else 0.0
    hist_acf = [_dep_autocorr(x, lag) for lag in (1, 2, 5, 10)]
    return {"ar8": (coef, refvar), "null": null, "hist_acf": hist_acf, "arch_hist": arch_hist}


def _dep_chow_supf(v):
    n = len(v)
    if n < 24:
        return 0.0, 0.0, 0.0
    csum = np.cumsum(v)
    csq = np.cumsum(v * v)
    tot = csum[-1]
    lo, hi = max(6, n // 8), n - max(6, n // 8)
    ks = np.arange(lo, hi, max(1, (hi - lo) // 40))
    best_t, best_v, best_pos = 0.0, 0.0, 0.0
    for k in ks:
        m1 = csum[k - 1] / k
        m2 = (tot - csum[k - 1]) / (n - k)
        v1 = max(csq[k - 1] / k - m1 * m1, _DEP_EPS)
        v2 = max((csq[-1] - csq[k - 1]) / (n - k) - m2 * m2, _DEP_EPS)
        t = abs(m1 - m2) / math.sqrt(v1 / k + v2 / (n - k) + _DEP_EPS)
        lvr = abs(math.log(v2 / v1))
        if t > best_t:
            best_t, best_pos = t, k / n
        if lvr > best_v:
            best_v = lvr
    return best_t, best_v, best_pos


def _dep_feats_full(w, dref):
    w = np.asarray(w, dtype=np.float64)
    coef, refvar = dref["ar8"]
    out = []
    r = _dep_ar_resid(w, coef, 8) if coef is not None else None
    if r is not None and len(r) >= 8:
        logvr = math.log((float(np.var(r)) + _DEP_EPS) / refvar)
        m, s = dref["null"]
        z_student = (logvr - m) / s
        rs = r / (math.sqrt(refvar) + _DEP_EPS)
        surprise = rs * rs - 1.0
        cusum = np.cumsum(surprise)
        nn = len(cusum)
        cmax = float(np.max(cusum)) / math.sqrt(nn)
        cend = float(cusum[-1]) / math.sqrt(nn)
        cpos = float(np.argmax(cusum)) / nn
        cdrift = float(cusum[-1] - cusum[0]) / nn
        arch_w = _dep_autocorr(r * r, 1)
    else:
        logvr = z_student = cmax = cend = cpos = cdrift = arch_w = 0.0
    out += [logvr, z_student, cmax, cend, cpos, cdrift]
    for lag, h_ac in zip((1, 2, 5, 10), dref["hist_acf"]):
        out.append(_dep_autocorr(w, lag) - h_ac)
    out.append(arch_w - dref["arch_hist"])
    t_sf, v_sf, pos_sf = _dep_chow_supf(w)
    out += [t_sf, v_sf, pos_sf]
    return out


# Chain onto the residual block's wrappers -- train()/infer() call these BY NAME (late binding),
# so appending here needs no change to the training or inference code.
_dep_base_hist_context = hist_context
_dep_base_compute_features_ms = compute_features_ms


def hist_context(x_h, hist_cap):
    ctx = _dep_base_hist_context(x_h, hist_cap)
    ctx["dep_ref"] = _dep_fit_refs(x_h)
    return ctx


def compute_features_ms(w, ctx):
    base = _dep_base_compute_features_ms(w, ctx)
    df = _dep_feats_full(np.asarray(w, dtype=np.float64), ctx["dep_ref"])
    return base + [df[i] for i in _DEP_KEEP]


print(f"dependence block (b_no_accf): +{len(_DEP_KEEP)} feats "
      f"(studentized resid + early-onset CUSUM + arch contrast + sup-F; ACF-lag block dropped)")


## `train()`

For each training series we compute `hist_context` once, sample `STEPS_PER_SERIES` online positions,
build the 15-dim feature vector for the trailing window at each, and label it `1` if that position is
at/after the break `tau` (else `0`). A `HistGradientBoostingClassifier` is fit on the pooled rows and
saved to disk.

In [ ]:
def train(
    datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
    model_directory_path: str,
):
    """Fit two decorrelated models on the SAME 45 multi-scale features:

    * a tuned HGB (pointwise, STEPS_PER_SERIES steps/series), and
    * an LGBM ranker (rank_xendcg, DENSE_K steps/series, query group = online step t).

    Each model's SORTED training-score vector is stored as its empirical-CDF reference so
    infer() can convert a live output to a fixed percentile without any cross-series state.
    """
    rng = np.random.default_rng(SEED)
    HAZ_S = 20.0                 # E72 hazard-head scale: soft target 1 - exp(-max(t-tau,0)/HAZ_S)
    hgb_rows: List[List[float]] = []
    hgb_off: List[int] = []      # per-row post-break offset o = t - tau (-1 if pre-break/no-break)
    rk_rows: List[List[float]] = []
    rk_labels: List[int] = []
    rk_t: List[int] = []

    n_used = 0
    for dataset_id, x_hist, x_online, tau in datasets:
        if n_used >= MAX_TRAIN_SERIES:
            break
        x_h = np.asarray(x_hist, dtype=np.float64)
        x_on = np.asarray(list(x_online), dtype=np.float64)  # materialize (iterable once)
        L = len(x_on)
        if L <= MIN_N or len(x_h) < 5:
            continue
        ctx = hist_context(x_h, HIST_CAP)
        cand = np.arange(MIN_N, L)

        # HGB pointwise rows (STEPS_PER_SERIES steps/series, matches the champion)
        kh = min(STEPS_PER_SERIES, len(cand))
        for t in rng.choice(cand, size=kh, replace=False):
            lo = max(0, t - WINDOW + 1)
            hgb_rows.append(compute_features_ms(x_on[lo:t + 1], ctx))
            hgb_off.append((t - tau) if (tau is not None and t >= tau) else -1)

        # Ranker rows (dense: DENSE_K steps/series), recording the group key (online step t)
        kr = min(DENSE_K, len(cand))
        for t in rng.choice(cand, size=kr, replace=False):
            lo = max(0, t - WINDOW + 1)
            rk_rows.append(compute_features_ms(x_on[lo:t + 1], ctx))
            rk_labels.append(1 if (tau is not None and t >= tau) else 0)
            rk_t.append(int(t))
        n_used += 1

    # --- tuned HGB champion (pointwise), E72 hazard-target head (regressor on soft target) ---
    X_hgb = np.asarray(hgb_rows, dtype=np.float64)
    y_hgb = 1.0 - np.exp(-np.clip(np.asarray(hgb_off, dtype=np.float64), 0, None) / HAZ_S)
    print(f"HGB matrix {X_hgb.shape} from {n_used} series, mean soft target {y_hgb.mean():.3f}")
    hgb = HistGradientBoostingRegressor(
        max_iter=MAX_ITER, learning_rate=0.021145795969329348, max_leaf_nodes=63,
        min_samples_leaf=10, l2_regularization=1.0, max_features=0.5,
        early_stopping=True, random_state=SEED,
    )
    hgb.fit(X_hgb, y_hgb)
    print(f"trained HGB {hgb.n_iter_} trees")
    ref_hgb = np.sort(hgb.predict(X_hgb))

    # --- LGBM listwise ranker (rank_xendcg): sort rows so each online step is a contiguous group ---
    X_rk = np.asarray(rk_rows, dtype=np.float64)
    y_rk = np.asarray(rk_labels, dtype=np.int64)
    t_rk = np.asarray(rk_t, dtype=np.int64)
    order = np.argsort(t_rk, kind="stable")
    X_rk, y_rk, t_rk = X_rk[order], y_rk[order], t_rk[order]
    _, groups = np.unique(t_rk, return_counts=True)
    print(f"ranker matrix {X_rk.shape}, {len(groups)} query groups (online steps)")
    n_jobs_fit = max(1, (os.cpu_count() or 4) - 1)
    ranker = lgb.LGBMRanker(n_jobs=n_jobs_fit, **LGB_PARAMS)
    with threadpool_limits(limits=1):
        ranker.fit(X_rk, y_rk, group=groups)
    ranker.set_params(n_jobs=1)  # single-row streaming predict at inference
    ref_rank = np.sort(ranker.predict(X_rk))

    joblib.dump(
        {
            "hgb": hgb,
            "ranker": ranker,
            "ref_hgb": ref_hgb,
            "ref_rank": ref_rank,
            "blend_w": BLEND_W,
            "config": {
                "WINDOW": WINDOW,
                "MS_SCALES": list(MS_SCALES),
                "MIN_N": MIN_N,
                "HIST_CAP": HIST_CAP,
                "PREDICT_EVERY": PREDICT_EVERY,
            },
        },
        os.path.join(model_directory_path, "model.joblib"),
    )

## `infer()`

A generator yielding exactly one score per online point. Per series we compute `hist_context` once,
then maintain a trailing window (O(1) add/evict). Every `PREDICT_EVERY` steps we recompute the
feature vector and the model's break probability; between refreshes the last score is held. Points
before `MIN_N` yield `0.0`.

In [ ]:
def infer(
    datasets: Iterable[Tuple[List[float], Iterable[float]]],
    model_directory_path: str,
):
    """One score per online point = fixed-CDF percentile blend of the two models.

    score = w * F_hgb(proba) + (1 - w) * F_rank(raw), where F_* are each model's empirical
    CDF from TRAINING scores (searchsorted / N). No cross-series or future info is used, so
    this is exactly deployable; TS-AUC only needs the within-step ordering.
    """
    saved = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    hgb = saved["hgb"]
    ranker = saved["ranker"]
    ref_hgb = np.asarray(saved["ref_hgb"], dtype=np.float64)
    ref_rank = np.asarray(saved["ref_rank"], dtype=np.float64)
    w = float(saved["blend_w"])
    cfg = saved["config"]
    window        = int(cfg["WINDOW"])
    min_n         = int(cfg["MIN_N"])
    hist_cap      = int(cfg["HIST_CAP"])
    predict_every = int(cfg["PREDICT_EVERY"])
    global MS_SCALES
    MS_SCALES = tuple(cfg.get("MS_SCALES", MS_SCALES))
    n_hgb = len(ref_hgb)
    n_rank = len(ref_rank)
    try:
        ranker.set_params(n_jobs=1)
    except Exception:
        pass

    yield  # Signal readiness to the runner.

    # Force single-threaded BLAS/OpenMP at call time (see the champion notebook for why).
    with threadpool_limits(limits=1):
        for x_historical, x_online in datasets:
            x_h = np.asarray(x_historical, dtype=np.float64)
            ctx = hist_context(x_h, hist_cap)

            buf = deque()
            score = 0.0
            for idx, point in enumerate(x_online):
                if len(buf) == window:
                    buf.popleft()
                buf.append(float(point))
                n = len(buf)

                if n < min_n:
                    yield 0.0
                    continue

                if idx % predict_every == 0:
                    feats = np.asarray(compute_features_ms(np.asarray(buf), ctx),
                                       dtype=np.float64).reshape(1, -1)
                    proba = float(hgb.predict(feats)[0])
                    raw = float(ranker.predict(feats)[0])
                    pct_hgb = float(np.searchsorted(ref_hgb, proba, side="right")) / n_hgb
                    pct_rank = float(np.searchsorted(ref_rank, raw, side="right")) / n_rank
                    score = w * pct_hgb + (1.0 - w) * pct_rank

                yield float(score)

In [ ]:
# @crunch/keep:on
# Each worker is pinned to 1 BLAS/OpenMP thread (see infer), so we can safely
# use most of the available vCPUs. Auto-scales to whatever machine you pick.
INFER_PARALLELISM = max(1, (os.cpu_count() or 4) - 1)
# INFER_PARALLELISM = 1  # set to 1 while debugging

## Local test + TS-AUC

Runs the same flow the platform will run, then computes the leaderboard metric locally.

In [ ]:
crunch_tools.test(
    # force_first_train=False,
    # no_determinism_check=True,
)

In [ ]:
# Compute local TS-AUC (same as the leaderboard, on the local test split).
prediction = pd.read_parquet("prediction/prediction.parquet")
y_test = pd.read_parquet("data/y_test.reduced.parquet")

merged = prediction.merge(y_test, how="left", left_index=True, right_index=True)
merged["time_online"] = merged.groupby("id").cumcount()

weighted_auc_sum = 0.0
total_weight = 0.0
for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values
    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue
    auc_t = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)
    weighted_auc_sum += weight * auc_t
    total_weight += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")

## Submit

On Colab, run the cell below. Locally, download this notebook and upload it to the platform, then create a run.

In [ ]:
Message = ""
crunch_tools.submit(message=Message)